In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM, GRU, Dense, Dropout, Masking
)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report


In [2]:
# Load processed 400-timestep dataset
DATA_PATH = "../data/processed/processed_400_timesteps.npz"

data = np.load(DATA_PATH, allow_pickle=True)

X = data["X"]        # (10000, 400, 240)
mask = data["mask"] # (10000, 400)
y = data["y"]

print("X shape:", X.shape)
print("Mask shape:", mask.shape)
print("y shape:", y.shape)


X shape: (10000, 400, 240)
Mask shape: (10000, 400)
y shape: (10000,)


In [3]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

num_classes = len(label_encoder.classes_)
print("Number of classes:", num_classes)


Number of classes: 2


In [4]:
from sklearn.model_selection import train_test_split
import numpy as np

# Create indices instead of copying data
indices = np.arange(len(y_encoded))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Train samples:", len(train_idx))
print("Validation samples:", len(val_idx))


Train samples: 8000
Validation samples: 2000


In [6]:
def disk_generator(npz_path, indices, batch_size=4):
    """
    Memory-safe generator that reads data directly from disk.
    """
    while True:
        np.random.shuffle(indices)

        with np.load(npz_path, allow_pickle=True) as data:
            X_disk = data["X"]
            y_disk = data["y"]

            for start in range(0, len(indices), batch_size):
                end = start + batch_size
                batch_idx = indices[start:end]

                X_batch = X_disk[batch_idx]      # small batch only
                y_batch = y_disk[batch_idx]

                yield X_batch, y_batch


In [7]:
from sklearn.model_selection import train_test_split
import numpy as np

labels = np.load(
    "../data/processed/processed_400_timesteps.npz",
    allow_pickle=True
)["y"]

indices = np.arange(len(labels))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=labels
)


In [8]:
BATCH_SIZE = 4   # SAFE even on 8 GB RAM

train_gen = disk_generator(
    "../data/processed/processed_400_timesteps.npz",
    train_idx,
    batch_size=BATCH_SIZE
)

val_gen = disk_generator(
    "../data/processed/processed_400_timesteps.npz",
    val_idx,
    batch_size=BATCH_SIZE
)

steps_per_epoch = len(train_idx) // BATCH_SIZE
val_steps = len(val_idx) // BATCH_SIZE


In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Masking

model_lstm = Sequential()
model_lstm.add(Masking(mask_value=0.0, input_shape=(400, 240)))
model_lstm.add(LSTM(32))                 # VERY IMPORTANT
model_lstm.add(Dense(num_classes, activation="softmax"))

model_lstm.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_lstm.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 masking (Masking)           (None, 400, 240)          0         
                                                                 
 lstm (LSTM)                 (None, 32)                34944     
                                                                 
 dense (Dense)               (None, 2)                 66        
                                                                 
Total params: 35,010
Trainable params: 35,010
Non-trainable params: 0
_________________________________________________________________


In [11]:
history = model_lstm.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    validation_steps=val_steps,
    epochs=5,
    verbose=1
)


StopIteration: 